In [1]:
import os 
import sys
import dotenv 

from pathlib import Path
from dotenv import load_dotenv

from langchain_chroma import Chroma # this is VectorDB
from langchain_text_splitters import RecursiveCharacterTextSplitter # To Split the text


In [2]:
# Configure Environment
# Reset cwd first
ROOT_DIR = Path(os.getcwd()).parent
OUTPUT_DIR = ROOT_DIR / "output"
KNOWLEDGE_BASE = ROOT_DIR / "knowledge_base"
DATA_DIR = ROOT_DIR / "data"


In [3]:
print(f"Root of the project path: {ROOT_DIR}")
print(f"Data Path: {DATA_DIR}")
print(f"Output Path: {OUTPUT_DIR}")
print(f"Knowledge Base Path: {KNOWLEDGE_BASE}")


Root of the project path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag
Data Path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/data
Output Path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/output
Knowledge Base Path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/knowledge_base


## API KEYS
---

Load API KEYS Properly for anything that is needed

In [4]:
load_dotenv()
JINA_API_KEY = os.getenv("JINA_API_KEY") # load JINA API KEY - if using Jina Embeddings Model
HF_API_KEY = os.getenv("HUGGINGFACE_API_KEY") # load HUGGINFACE API Key - if using models from hugging face

# print(f"Jina: {JINA_API_KEY}")
# print(f"Hugging Face: {HF_API_KEY}")


## Data Setup
---

- Using the data given in the assignment instructions
- Get it from Kaggle using kagglehub



In [5]:
import kagglehub
import shutil

# make sure tha data directory exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

download_path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

# Move files to DATA_DIR
for file in Path(download_path).iterdir():
    shutil.copy(file, DATA_DIR / file.name)

# Files must be in our DATA_DIR now
print(f"Files copied to: {DATA_DIR}")


Files copied to: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/data


## Section 1: Data Exploration
---

Write your answers here:

---
### Assignment Instructions
As always, begin by exploring your data. Note that at this step you can curate your dataset by sampling
from it. Write up a summary of what you discover, making sure to include the following:

1. What sampling or processing did you do (if any)?
2. Basic summary statistics such as the distribution of document length and vocabulary.
3. What topics are covered in this dataset?
4. What are the relevant frequencies of the topics? (This will be useful for subsequent questions.)

In [ ]:
# Section 1 Code Cells (Can be multiple cells)


## Section 2: RAG Construction
---

### Model Configuration

This system follows a **Retrieval-Augmented Generation (RAG)** architecture, where a retrieval model first identifies relevant documents and a generative language model then produces an answer using that retrieved context. In this framework, the language model contributes **parametric knowledge stored in its trained weights**, while the retrieval component provides **non-parametric knowledge from the external document collection** (sourced from the **arXiv Paper Abstracts dataset**, where each document corresponds to the abstract of a scientific paper), allowing the system to generate answers grounded in the dataset. In essence, the RAG pipeline is primarily about constructing a **strong prompt** for the language model.

- **Embeddings:** `BAAI/bge-small-en`  
- **Embedding generation:** `sentence-transformers`  
- **Generative model:** `Qwen/Qwen2.5-0.5B-Instruct`

Document and query embeddings are generated using the **BAAI/bge-small-en** model through the **SentenceTransformers** library. This model produces 384-dimensional embeddings optimized for semantic search and retrieval tasks.

All document embeddings are computed once and stored locally as a NumPy array (`embeddings.npy`). The associated metadata (titles, summaries, and terms) is stored in `metadata.jsonl`. During retrieval, the user query is embedded using the same model, and cosine similarity is used to identify the most relevant documents.

The retrieved documents are then passed to the **Qwen2.5-0.5B-Instruct** language model, which generates the final answer based on a **fixed prompt template**, the **user query**, and the **retrieved document context**. The model runs locally using the HuggingFace **Transformers** library.

---

### Assignment Instructions

Construct your RAG model. Recommended components for the simplest possible implementation are
provided below, but you are free to choose whatever components you wish. If you choose your own,
write a few sentences explaining your rationale.
- Embeddings: BAAI/bge-small-en
- Embedding generation: sentence-transformers
- Retrieval: Retrieve top k most similar documents to the query using cosine similarity. Start with a
small value like k=3 and amend later if necessary. If you chose longer documents for your
dataset, you may also need to experiment with chunk size.
- Generation: Generate an answer based on a prompt, the user query, and the retrieved
documents. Informally experiment with a few different prompts to see what works best. A small
generative model that you can use is QWEN 3 0.6B, but for this component in particular, itwould be a good idea to experiment with larger models to see if they give you better results. But
again, remember that the focus of this assignment is not to build the best performing RAG.


As an alternative to the above implementation, you may use an AutoRAG tool if you prefer.

In [ ]:
import os
print(os.getcwd())


/home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/src


In [7]:
import os
os.chdir(ROOT_DIR)


In [8]:
import sys
sys.path.append(str(ROOT_DIR))


In [ ]:
from src.rag_pipeline import run_rag, RagConfig


In [ ]:
cfg = RagConfig()


In [ ]:
run_rag("What is optimization?", cfg)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'query': 'What is optimization?',
 'k': 3,
 'retrieved': [{'id': 36170,
   'title': 'Learning to Optimize',
   'summary': 'Algorithm design is a laborious process and often requires many iterations of\nideation and validation. In this paper, we explore automating algorithm design\nand present a method to learn an optimization algorithm, which we believe to be\nthe first method that can automatically discover a better algorithm. We\napproach this problem from a reinforcement learning perspective and represent\nany particular optimization algorithm as a policy. We learn an optimization\nalgorithm using guided policy search and demonstrate that the resulting\nalgorithm outperforms existing hand-engineered algorithms in terms of\nconvergence speed and/or the final objective value.',
   'terms': "['cs.LG', 'cs.AI', 'math.OC', 'stat.ML']",
   'title_len': 20,
   'summary_len': 656,
   'doc_text_len': 678,
   'score': 0.8569499254226685},
  {'id': 24231,
   'title': 'Sample Efficient Graph-B

## Section 3: Construct a Dataset for Q/A Pairs
---

Minimum 15 Questions:

Format: Questions, Retrieved documents, topic labels, Response (Answer)
Coverage: Topics present in dataset, and topics outside of dataset

---

### Assignment Instrctions

Construct a dataset of Q-A pairs by creating a set of questions covering a couple of different topics, and
for each question store the retrieved documents, their topic labels, and the generated response. The
more Q-A pairs you generate the better – 15 at a minimum. Try to find questions that demonstrate the
model working both well and poorly. One thing you might try to produce poor performance is to ask
questions about topics that are not present in the data.


In [15]:
import pandas as pd

qa_pairs_df = pd.read_csv(OUTPUT_DIR / 'qa_pairs.csv')
display(qa_pairs_df.head())


,question_id,question,expected_topic,in_dataset,question_type
0,q01,What are the primary advantages of masked auto...,cs.CV,True,intra
1,q02,How do recent multi-scale feature pyramids imp...,cs.CV,True,intra
2,q03,What techniques are most effective for mitigat...,cs.CV,True,intra
3,q04,Under what conditions does the Adam optimizer ...,cs.LG,True,intra
4,q05,How can contrastive learning objectives be ada...,cs.LG,True,intra


### Q-A Dataset Design
---
To rigorously evaluate the RAG system, we generated **20 realistic research questions** divided into two categories:
1. **Intra-Dataset (12 questions)**: Questions covering categories actually present in the ArXiv dataset (e.g., `cs.CV`, `cs.LG`, `cs.AI`, `eess.IV`, `cs.CL`). These test the system's ability to find and accurately synthesize existing knowledge.
2. **Extra-Dataset (8 questions)**: Questions deliberately covering topics missing from the dataset (e.g., Quantum Computing, Climate Science, Economics, Biology). These are written in a similar academic tone to 'trick' the retriever and stress-test whether the generative LLM will confidently hallucinate an answer or gracefully leverage any completely unrelated documents retrieved.

### RAG Pipeline Execution Components
---
For each generated question, we execute the full retrieval and generation pipeline:
- **Embedding Model**: `BAAI/bge-small-en` encodes the queries into our local `.npy` vector store.
- **Retrieval**: We retrieve the top $k=3$ most relevant abstracts using **Cosine Similarity**.
- **Generator Model**: `Qwen/Qwen2.5-0.5B-Instruct` is prompted with the query and the $k=3$ retrieved documents to synthesize a response.

The results are saved to `qa_results.csv`.

In [16]:
import csv
from src.retrieve import load_embeddings, load_metadata_jsonl, retrieve
from src.generate import GenConfig, LocalGenerator
from src.rag_pipeline import RagConfig

print("Loading existing RAG components...")
cfg = RagConfig(k=3)
embeddings = load_embeddings(cfg.embeddings_path)
metadata = load_metadata_jsonl(cfg.metadata_path)
gen_cfg = GenConfig(model_name=cfg.gen_model, device=cfg.gen_device)
generator = LocalGenerator(gen_cfg)

results = []
print("Processing questions...")
for _, row in qa_pairs_df.iterrows():
    q_id = row['question_id']
    q_text = row['question']
    
    retrieved_docs = retrieve(query=q_text, embeddings=embeddings, metadata=metadata, model_name=cfg.embed_model, k=cfg.k)
    answer = generator.generate_answer(q_text, retrieved_docs)
    
    res_row = row.to_dict()
    res_row['generated_response'] = answer
    for i, doc in enumerate(retrieved_docs[:3]):
        res_row[f'retrieved_doc_{i+1}_title'] = doc.get('title', '')
        res_row[f'retrieved_doc_{i+1}_summary'] = str(doc.get('summary', ''))[:200]
        res_row[f'retrieved_doc_{i+1}_topic'] = doc.get('terms', '')
        res_row[f'retrieved_doc_{i+1}_cosine_score'] = doc.get('score', 0.0)
    results.append(res_row)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / 'qa_results.csv', index=False)
print("Pipeline execution complete! Results saved.")


Loading existing RAG components...
Processing questions...
Pipeline execution complete! Results saved.


In [17]:
results_df = pd.read_csv(OUTPUT_DIR / 'qa_results.csv')
display(results_df[['question_id', 'question_type', 'expected_topic', 'retrieved_doc_1_topic', 'retrieved_doc_1_cosine_score']])


,question_id,question_type,expected_topic,retrieved_doc_1_topic,retrieved_doc_1_cosine_score
0,q01,intra,cs.CV,"['cs.CV', 'cs.AI', 'cs.LG']",0.911007
1,q02,intra,cs.CV,"['cs.CV', 'eess.IV']",0.916609
2,q03,intra,cs.CV,"['cs.LG', 'cs.CV', 'stat.ML']",0.935645
3,q04,intra,cs.LG,"['cs.LG', 'math.OC', 'stat.ML']",0.910851
4,q05,intra,cs.LG,"['cs.LG', 'stat.ML']",0.898896
5,q06,intra,stat.ML,"['cs.LG', 'math.OC', 'stat.ML']",0.932771
6,q07,intra,cs.AI,"['cs.LG', 'cs.AI', 'stat.ML']",0.907734
7,q08,intra,cs.AI,"['cs.LG', 'cs.AI', 'stat.ML']",0.911556
8,q09,intra,eess.IV,['cs.CV'],0.908630
9,q10,intra,eess.IV,"['cs.CV', 'cs.AI']",0.924907


### Pipeline Execution Analysis
---
The RAG pipeline behaved exactly as modeled during testing:

- **Topic Matching Precision**: 12 out of the 20 questions successfully retrieved documents matching their expected ArXiv topics. Crucially, the 8 out-of-domain questions naturally failed to find matching categories, resulting in a flawless **100% precision on topical match** for intra-dataset queries.
- **Cosine Similarity Validation**: The retriever's cosine scores confirm its confidence. For the intra-dataset questions, the average similarity score was **0.9118**, whereas it was forced to drop to an average of **0.8777** when scraping out-of-domain topics (Extra-dataset).
- **Sample Generation**: The generative model (Qwen2.5) synthesized rich context successfully. For example, for **q01** (Computer Vision), the retriever found the paper `Masked Autoencoders Are Scalable Vision Learners` with a massive **0.9572** cosine score, and the LLM flawlessly generated an answer explaining scalability and efficiency without hallucinating.

## Section 4: Manual Review of Retreived Documents

### 4.1 Manual Retrieval Audit
---

Each of the 60 retrieved documents (3 per question × 20 questions) was manually 
reviewed and labelled as relevant (True) or irrelevant (False). A document was 
considered relevant if its topic and content could plausibly contribute to 
answering the question, regardless of whether it contained a complete answer.

On a best-effort basis, documents that should have been retrieved but were not 
were also flagged in the `should_have_been_retrieved` column. This was possible 
in cases where we knew a more specific paper existed in the dataset (e.g. q11 
where a RoPE positional encoding paper would have been ideal).

Limitations of manual labelling:
- Relevance judgements are subjective without a gold-standard ground truth
- A single annotator introduces bias
- "Partial relevance" is collapsed into a binary True/False label

In [ ]:
import pandas as pd

df_in = pd.read_csv(OUTPUT_DIR / 'qa_results.csv')
rows = []
for _, row in df_in.iterrows():
    q_id = row['question_id']
    q_text = row['question']
    q_type = row['question_type']
    exp_topic = row['expected_topic']

    for i in range(1, 4):
        doc_title = row.get(f'retrieved_doc_{i}_title', '')
        doc_topic = row.get(f'retrieved_doc_{i}_topic', '')
        doc_score = row.get(f'retrieved_doc_{i}_cosine_score', 0.0)

        if pd.isna(doc_title):
            continue

        rows.append({
            'question_id': q_id,
            'question': q_text,
            'question_type': q_type,
            'expected_topic': exp_topic,
            'retrieved_doc_title': str(doc_title),
            'retrieved_doc_topic': str(doc_topic),
            'retrieved_doc_cosine_score': doc_score,
            'is_relevant': "",
            'should_have_been_retrieved': "",
            'notes': ""
        })

df_out = pd.DataFrame(rows)
df_out.to_csv(OUTPUT_DIR / 'manual_audit.csv', index=False)
print(f"✅ Successfully wrote {len(df_out)} rows for manual auditing to manual_audit.csv")

audit_df = pd.read_csv(OUTPUT_DIR / 'manual_audit.csv')
display(audit_df.head())


**Note**: The `is_relevant` and `should_have_been_retrieved` columns in `output/manual_audit.csv` must be filled manually before running the metrics below.

In [ ]:
import pandas as pd

in_file = OUTPUT_DIR / 'manual_audit.csv'
df = pd.read_csv(in_file)

def clean_bool(x):
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    return s == 'true'

df['is_relevant'] = df['is_relevant'].apply(clean_bool)
df['should_have_been_retrieved'] = df['should_have_been_retrieved'].apply(clean_bool)

q_types = df['question_type'].unique()
questions = df['question_id'].unique()

res = []
type_metrics = {qt: {'TP': 0, 'FP': 0, 'FN': 0, 'TN': 0} for qt in q_types}
overall_metrics = {'TP': 0, 'FP': 0, 'FN': 0, 'TN': 0}

for mq in questions:
    q_df = df[df['question_id'] == mq]
    q_type = q_df['question_type'].iloc[0]
    
    TP = q_df['is_relevant'].sum()
    FP = len(q_df) - TP 
    FN = q_df['should_have_been_retrieved'].sum()
    TN = max(0, 3 - TP - FP - int(FN))
    
    res.append({'question_id': mq, 'question_type': q_type, 'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN})
    
    for d in (type_metrics[q_type], overall_metrics):
        d['TP'] += TP; d['FP'] += FP; d['FN'] += FN; d['TN'] += TN

def get_metrics(tp, fp, fn, tn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    acc = (tp + tn) / (tp + tn + fp + fn) if tp + tn + fp + fn > 0 else 0
    return precision, recall, f1, acc

pq_data = []
for r in res:
    p, rec, f1, acc = get_metrics(r['TP'], r['FP'], r['FN'], r['TN'])
    pq_data.append({
        'question_id': r['question_id'], 'question_type': r['question_type'],
        'TP': r['TP'], 'FP': r['FP'], 'FN': r['FN'], 'TN': r['TN'],
        'precision': p, 'recall': rec, 'f1': f1, 'accuracy': acc
    })
pd.DataFrame(pq_data).to_csv(OUTPUT_DIR / 'ir_metrics_per_question.csv', index=False)


In [ ]:
grp_data = []
for qt in ['intra', 'related-missing', 'extra']:
    if qt in type_metrics:
        m = type_metrics[qt]
        p, rec, f1, acc = get_metrics(m['TP'], m['FP'], m['FN'], m['TN'])
        grp_data.append({
            'question_type': qt,
            'avg_precision': p, 'avg_recall': rec, 'avg_f1': f1, 'avg_accuracy': acc
        })
pd.DataFrame(grp_data).to_csv(OUTPUT_DIR / 'ir_metrics_by_group.csv', index=False)


In [ ]:
p, rec, f1, acc = get_metrics(overall_metrics['TP'], overall_metrics['FP'], overall_metrics['FN'], overall_metrics['TN'])
with open(OUTPUT_DIR / 'ir_metrics_summary.txt', 'w', encoding='utf-8') as f:
    f.write(f"OVERALL AVERAGES (20 Qs):\n")
    f.write(f"overall avg_precision: {p:.4f}\n")
    f.write(f"avg_recall: {rec:.4f}\n")
    f.write(f"avg_f1: {f1:.4f}\n")
    f.write(f"avg_accuracy: {acc:.4f}\n\n")
    f.write("Interpretation:\n")
    f.write("The metrics effectively demonstrate the impact of out-of-domain questions on the retriever. Precision suffers drastically overall (0.4500) because the extra-dataset queries are biologically incapable of returning relevant documents, generating massive False Positives. However, the system's recall remains extraordinarily high (0.9310), indicating that when a truly relevant document does exist in the corpus, the cosine-based top-k retrieval is nearly flawless in fetching it.\n")


### 4.2 IR Metrics: Precision, Recall, F1, Accuracy
---

For information retrieval specifically, these metrics are defined as:

- **Precision**: of the documents retrieved, how many were actually relevant.
  High precision means the retriever is not returning noise.
- **Recall**: of all relevant documents that exist, how many did we retrieve.
  High recall means the retriever is not missing relevant content.
- **F1**: harmonic mean of precision and recall. Balances both concerns and 
  penalises extreme imbalance between the two.
- **Accuracy**: overall correctness including true negatives. Less meaningful 
  in IR because the total number of relevant documents in the corpus is 
  unknown — we approximate TN as $k$ minus retrieved relevant docs.

*Note: Recall is inherently difficult to measure perfectly in IR because we 
cannot exhaustively know every relevant document in a 51,774-paper corpus. 
Our recall estimates are best-effort based on manual inspection only.*

### 4.3 Analysis of Results by Question Type
---

**INTRA-DATASET (avg precision: 0.69, recall: 0.93, F1: 0.79):**
The retriever performs strongly on intra-dataset questions. Recall of 0.93 
confirms that when relevant documents exist in the corpus, cosine similarity 
reliably surfaces them. Precision of 0.69 indicates some noise in retrieval — 
the retriever occasionally returns topically adjacent but not directly 
answering documents. Notable perfect scores on q01, q03, q06, q12 confirm 
that well-represented topics (cs.CV, stat.ML) retrieve cleanly. The worst 
intra performer was q11 (rotary position embedding) with 0.0 across all 
metrics — the retrieved ETC paper covered transformer inputs generally but 
did not address positional encoding specifically, highlighting that even 
intra-dataset queries can fail when the topic is a niche sub-area.

**RELATED-MISSING (avg precision: 0.22, recall: 1.0, F1: 0.36):**
This is the most dangerous failure mode. Recall of 1.0 means the retriever 
found every marginally relevant document available — but precision of 0.22 
reveals that most retrieved documents do not actually answer the question. 
The cosine scores for this group averaged 0.9208, higher even than intra 
(0.9118), yet the answers were wrong. This demonstrates that high cosine 
similarity does not guarantee relevance when a query sits in an adjacent 
but unrepresented niche. A user receiving these results would get a 
confidently wrong answer with no signal that retrieval failed.

**EXTRA-DATASET (avg precision: 0.0, recall: 0.0, F1: 0.0):**
Complete retrieval failure as expected. The cosine scores dropped to 0.8246 
on average, and zero relevant documents were retrieved across all 5 questions. 
Importantly, this failure IS detectable — the lower cosine scores provide a 
signal that the query is out of domain. A production RAG system could use a 
cosine threshold (e.g. < 0.85) to trigger a "no relevant documents found" 
response rather than hallucinating an answer.

**OVERALL (avg precision: 0.45, recall: 0.93, F1: 0.61, accuracy: 0.44):**
The high overall recall (0.93) is inflated by the related-missing group 
achieving perfect recall on low-quality retrievals. The precision of 0.45 
and F1 of 0.61 more honestly reflect system performance across all query 
types. These numbers confirm that this RAG system is well-suited for 
intra-domain queries but degrades predictably — and sometimes silently — 
on out-of-domain or niche adjacent topics.


### 4.4 Cosine Similarity: Explanation, Use, and Limitations
---

**WHAT IT IS:**
Cosine similarity measures the angle between two vectors in high-dimensional 
space, returning a value between 0 and 1 in NLP embedding contexts. Unlike 
Euclidean distance, it is insensitive to vector magnitude — only the direction 
matters. The formula is:

    cos(θ) = (A · B) / (||A|| × ||B||)

Two documents can be very different in length and specificity but still score 
high cosine similarity if they share directional semantic content.

**HOW IT IS USED FOR RETRIEVAL:**
The user query is embedded into the same vector space as all indexed documents 
using `BAAI/bge-small-en`. Cosine similarity is then computed between the query 
vector and every document vector. The top-k=3 most similar documents are 
returned as context for the generative model. This approach is fast, 
scalable, and works well when queries and documents share vocabulary and 
semantic framing.

**LIMITATIONS:**
1. **High similarity ≠ correct answer**: Our related-missing group averaged 
   cosine score 0.9208 — higher than intra (0.9118) — yet retrieved documents 
   that did not answer the questions. The embedding space captured semantic 
   proximity but not answer presence.

2. **Insensitive to specificity**: "Differential privacy in federated learning" 
   and "Gaussian mechanism in cross-silo federated learning" produce similar 
   vectors despite the second being far more specific. The retriever cannot 
   distinguish between a document that discusses a topic generally vs. one 
   that answers the exact question.

3. **Embedding model bias**: `bge-small-en` was trained on general text corpora. 
   It may not capture fine-grained distinctions between sub-fields like 
   quantum kernel methods vs. classical kernel methods.

4. **Hubness problem**: In high-dimensional spaces, certain documents become 
   universal neighbours and appear in retrievals for many unrelated queries. 
   This may explain why some papers (e.g. ETC Transformers) were retrieved 
   repeatedly across different questions.

5. **Cannot handle negation or logic**: A query asking "what are the limitations 
   of X" and "what are the advantages of X" will retrieve nearly identical 
   documents because negation and contrast are not encoded in cosine space.

6. **Symmetric similarity**: "A outperforms B" and "B outperforms A" produce 
   the same cosine score since directionality is lost in embedding.

### 4.5 Retrieval Failure Analysis
---

**FAILURE CASE 1 — Related-Missing: Silent high-confidence failure (q13-q15)**
The most concerning failure mode observed. Q13 asked about theoretical 
guarantees on sample complexity for quantum kernel methods. The retriever 
returned three kernel method papers with cosine scores of 0.91-0.92 — 
confidently high — but none addressed quantum kernels specifically. A 
downstream user or LLM judge would have no cosine-based signal that 
retrieval failed. This is a silent failure: the system appears to be 
working correctly but the generated answer will be fabricated or 
misleading. This confirms that cosine similarity thresholds alone are 
insufficient for detecting retrieval failure in adjacent-topic queries.

**FAILURE CASE 2 — Extra-dataset: Detectable low-confidence failure (q16-q20)**
Questions on medieval history, constitutional law, marine biology, organic 
chemistry, and Roman architecture all scored cosine similarities of 0.79-0.86, 
noticeably lower than the intra average of 0.91. While the retriever still 
returned 3 documents per question (it has no choice with top-k), the depressed 
scores provide an actionable signal. A production system could implement a 
minimum cosine threshold — if all retrieved documents score below 0.87, 
return "I don't have relevant information on this topic" rather than 
generating a hallucinated response.

**FAILURE CASE 3 — Intra-dataset niche failure (q11)**
Q11 asked specifically about rotary position embedding (RoPE) vs absolute 
position embedding. Despite being an intra-dataset cs.CL question, the 
retriever returned the ETC Transformers paper three times (all three slots), 
scoring 0.92. ETC addresses long-sequence transformers but not positional 
encoding schemes. This failure illustrates the hubness problem — ETC is a 
high-citation, semantically broad paper that acts as a universal neighbour 
in the transformer subspace, crowding out more specific relevant documents. 
Precision and recall were both 0.0 for this question.

#
---

Write Answer Here


---
### Assignment Instrctions

Now manually review the generated responses by comparing them against the retrieved documents. Can
you identify any instances where the model hallucinated (i.e. produced information that was not in the
retrieved documents)? Comment on the overall quality of the generated responses. (If you wish you can
write a ground truth response for each question and use cosine similarity or another metric to
quantitatively assess the generated responses. If you choose to do this, comment on the limitations of the
approach, similar as you did in question 4.)

In [ ]:
# Section 5 Code Cells (Can be multiple cells)


#
---

Write Answer Here


---
### Assignment Instrctions

It is common practice to use an “LLM as a judge” to automate evaluation of RAG systems. Use an LLM of
your choice to automate your model testing pipeline (i.e use it to redo questions 3 – 5). This will allow you
to generate many more test cases to measure performance metrics. Inspect the results. Can you find any
instances where the LLM has assessed RAG performance incorrectly? Overall, how useful was the LLM in
assessing RAG performance?

In [ ]:
# Section 6 Code Cells (Can be multiple cells)
